# الفصل الرابع عشر: البرمجة الكائنية

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-14.ipynb)

> **الوقت المتوقع:** ~8 دقيقة قراءة · **11** قسمًا · **27** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **14.1** لماذا الوراثة أحد أهم اختراعات البرمجة الكائنية؟
- **14.2** الوراثة (Inheritance) بعمق
- **14.3** تعدد الأشكال (Polymorphism)
- **14.4** الفئات المجردة (Abstract Base Classes)
- **14.5** Dataclasses: اختصار كتابة الفئات
- **14.6** مزيد من Magic Methods: `__call__`, `__getitem__`, Context Managers
- **14.7** مشروع تطبيقي متكامل: نظام حجز مكتبة
- **14.8** تمارين
- **14.9** اختبر فهمك (Quiz)
- **14.10** ملخص الفصل (Cheat Sheet)
- **14.11** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

## 14.1 لماذا الوراثة أحد أهم اختراعات البرمجة الكائنية؟

تخيّل أنك تبني نظاماً لمتجر يبيع منتجات متنوعة: كتب، إلكترونيات، ملابس. كل هذه المنتجات تشترك في خصائص أساسية (اسم، سعر، رقم تعريف)، لكن لكل نوع خصائص وسلوك إضافي خاص به (الكتاب له مؤلف، الإلكترونيات لها ضمان، الملابس لها مقاس).

بدون الوراثة، ستكتب ثلاث فئات منفصلة تماماً، كل واحدة تُكرر نفس كود `name`, `price`, `id`. هذا يخالف أحد أهم مبادئ البرمجة: **DRY (Don't Repeat Yourself)**.

**الوراثة** تحل هذه المشكلة بأناقة: تُعرّف فئة «أساسية» (Parent/Base Class) تحتوي الخصائص المشتركة، ثم تُنشئ فئات «فرعية» (Child/Derived Classes) تَرِث كل شيء من الفئة الأساسية تلقائياً، وتُضيف فقط ما يميزها.

In [ ]:
class Product:
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

    def display_info(self) -> str:
        return f"{self.name}: {self.price} ر.س"

class Book(Product):  # Book ترث من Product
    def __init__(self, name: str, price: float, author: str):
        super().__init__(name, price)  # استدعاء __init__ من الفئة الأساسية
        self.author = author

    def display_info(self) -> str:
        return f"{super().display_info()} - تأليف: {self.author}"

book = Book("بايثون للمحترفين", 120, "محمد أحمد")
print(book.display_info())
# بايثون للمحترفين: 120 ر.س - تأليف: محمد أحمد

في هذا الفصل، ستتعمق في الوراثة، وتعدد الأشكال (Polymorphism)، والفئات المجردة (Abstract Classes)، وأداة ثورية تُسمى **Dataclasses** تختصر عليك كتابة أكواد متكررة، ثم نختم بمشروع متكامل يجمع كل ما تعلمته في الفصلين الأخيرين.

## 14.2 الوراثة (Inheritance) بعمق

### الصيغة الأساسية

In [ ]:
class Animal:  # Parent / Base / Super Class
    def __init__(self, name: str):
        self.name = name

    def make_sound(self) -> str:
        return "صوت ما..."

    def __str__(self) -> str:
        return f"{self.name} يصدر صوت: {self.make_sound()}"

class Dog(Animal):  # Child / Derived / Sub Class
    def make_sound(self) -> str:  # Override - استبدال السلوك
        return "واق واق!"

class Cat(Animal):
    def make_sound(self) -> str:
        return "مياو!"

dog = Dog("ريكس")
cat = Cat("لونا")

print(dog)  # ريكس يصدر صوت: واق واق!
print(cat)  # لونا يصدر صوت: مياو!

لاحظ أن `Dog` و`Cat` لم تُعرّفا `__init__` و`__str__` من جديد — ورثتاها كاملة من `Animal`، وغيّرتا فقط `make_sound()`.

### دالة `super()`: الوصول للفئة الأساسية

In [ ]:
class Employee:
    def __init__(self, name: str, salary: float):
        self.name = name
        self.salary = salary

    def display(self) -> str:
        return f"{self.name}: {self.salary} ر.س"

class Manager(Employee):
    def __init__(self, name: str, salary: float, team_size: int):
        super().__init__(name, salary)  # استدعاء __init__ الأساسي أولاً
        self.team_size = team_size

    def display(self) -> str:
        base_info = super().display()  # استدعاء display الأساسي، ثم إضافة عليه
        return f"{base_info} (يدير فريق من {self.team_size} أشخاص)"

manager = Manager("نورة", 15000, 8)
print(manager.display())
# نورة: 15000 ر.س (يدير فريق من 8 أشخاص)

> ⚠️ **تحذير شائع جداً**: نسيان استدعاء `super().__init__()` عند تخصيص `__init__` في فئة فرعية:

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    class Manager(Employee):
        def __init__(self, name, salary, team_size):
            # ⚠️ نسيان super().__init__(name, salary)!
            self.team_size = team_size

    manager = Manager("نورة", 15000, 8)
    print(manager.name)  
    # 💥 AttributeError: 'Manager' object has no attribute 'name'
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

بدون استدعاء `super().__init__()`، لن تُهيَّأ الخصائص المُعرَّفة في الفئة الأساسية إطلاقاً.

> 💡 **نصيحة الخبراء**: استدعِ `super().__init__()` عادة **كأول سطر** في `__init__` الفئة الفرعية، لضمان تهيئة كل الخصائص الموروثة قبل إضافة أي خصائص جديدة.

### التحقق من العلاقات: `isinstance()` و `issubclass()`

In [ ]:
print(isinstance(dog, Dog))     # True
print(isinstance(dog, Animal))  # True - Dog هي أيضاً Animal!
print(isinstance(dog, Cat))     # False

print(issubclass(Dog, Animal))  # True
print(issubclass(Animal, Dog))  # False - العلاقة أحادية الاتجاه

> 💡 **نصيحة الخبراء**: استخدم `isinstance()` دائماً بدلاً من `type(obj) == SomeClass` للتحقق من نوع كائن، لأن `isinstance()` تأخذ الوراثة بعين الاعتبار:

```python
# ❌ لا يأخذ الوراثة في الحسبان
if type(dog) == Animal:  # False حتى لو dog هو كلب (نوع فرعي من Animal)!
    pass

# ✅ يأخذ الوراثة في الحسبان
if isinstance(dog, Animal):  # True
    pass
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### الوراثة متعددة المستويات (Multi-Level Inheritance)

In [ ]:
class Vehicle:
    def __init__(self, brand: str):
        self.brand = brand

    def describe(self) -> str:
        return f"مركبة من {self.brand}"

class Car(Vehicle):
    def __init__(self, brand: str, doors: int):
        super().__init__(brand)
        self.doors = doors

    def describe(self) -> str:
        return f"{super().describe()}, عدد الأبواب: {self.doors}"

class SportsCar(Car):
    def __init__(self, brand: str, doors: int, top_speed: int):
        super().__init__(brand, doors)
        self.top_speed = top_speed

    def describe(self) -> str:
        return f"{super().describe()}, السرعة القصوى: {self.top_speed} كم/س"

ferrari = SportsCar("Ferrari", 2, 340)
print(ferrari.describe())
# مركبة من Ferrari, عدد الأبواب: 2, السرعة القصوى: 340 كم/س

print(isinstance(ferrari, Vehicle))   # True
print(isinstance(ferrari, Car))       # True
print(isinstance(ferrari, SportsCar)) # True

### الوراثة المتعددة (Multiple Inheritance) و MRO

Python تسمح بوراثة فئة من أكثر من فئة أساسية واحدة:

In [ ]:
class Swimmer:
    def swim(self) -> str:
        return "يسبح 🏊"

class Flyer:
    def fly(self) -> str:
        return "يطير 🦅"

class Duck(Swimmer, Flyer):  # وراثة من فئتين معاً
    def __init__(self, name: str):
        self.name = name

duck = Duck("دونالد")
print(duck.swim())  # يسبح 🏊
print(duck.fly())   # يطير 🦅

> ⚠️ **تحذير احترافي**: الوراثة المتعددة قوية لكنها قد تُسبب تعقيداً كبيراً، خاصة عند تعارض الأسماء بين الفئات الأساسية. Python تحل هذا عبر خوارزمية تُسمى **MRO (Method Resolution Order)**:

In [ ]:
print(Duck.__mro__)
# (<class 'Duck'>, <class 'Swimmer'>, <class 'Flyer'>, <class 'object'>)

هذا يُحدد ترتيب البحث عن أي method أو attribute: Python تبحث أولاً في `Duck`، ثم `Swimmer`، ثم `Flyer`، ثم `object` (الفئة الأساسية لكل شيء في Python).

> 💡 **نصيحة الخبراء**: تجنّب الوراثة المتعددة المعقدة قدر الإمكان في مشاريعك الخاصة؛ استخدمها فقط لحالات بسيطة وواضحة (تُسمى أحياناً **Mixins** — فئات صغيرة تُضيف سلوكاً محدداً). إذا وجدت نفسك تحتاج هرمية وراثة معقدة جداً، فكّر في **Composition** (تضمين كائن داخل آخر) كبديل أنظف غالباً — مبدأ معروف باسم "Composition over Inheritance".

## 14.3 تعدد الأشكال (Polymorphism)

### الفكرة الأساسية

**Polymorphism** (تعدد الأشكال) يعني: كائنات من فئات مختلفة يمكن التعامل معها بواجهة موحدة، لكن كل واحدة تتصرف بطريقتها الخاصة.

In [ ]:
class Shape:
    def area(self) -> float:
        raise NotImplementedError("يجب تخصيص هذه الدالة في الفئة الفرعية")

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

class Triangle(Shape):
    def __init__(self, base: float, height: float):
        self.base = base
        self.height = height

    def area(self) -> float:
        return 0.5 * self.base * self.height

# هنا يكمن سحر تعدد الأشكال:
shapes: list[Shape] = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]

for shape in shapes:
    print(f"{type(shape).__name__}: المساحة = {shape.area():.2f}")

**الناتج:**

```
Circle: المساحة = 78.54
Rectangle: المساحة = 24.00
Triangle: المساحة = 12.00
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `المساحة` من مثال سابق.

**لاحظ القوة هنا**: الكود `shape.area()` **لا يعرف ولا يهتم** بنوع الشكل الفعلي أثناء كتابته. كل كائن «يعرف» كيف يحسب مساحته بطريقته الخاصة، والكود الذي يستخدمها يبقى موحداً وبسيطاً، بغض النظر عن إضافة أشكال جديدة لاحقاً.

> 💡 **نصيحة الخبراء**: هذا هو جوهر مبدأ **Open/Closed Principle** (أحد مبادئ SOLID التي سنلمسها أكثر في الفصل 24): الكود **مفتوح للتوسعة** (يمكنك إضافة `Pentagon` جديدة بسهولة) لكنه **مغلق للتعديل** (لا تحتاج تعديل حلقة `for` الموجودة أعلاه عند إضافة شكل جديد).

### Duck Typing: فلسفة Python الفريدة

In [ ]:
class Duck:
    def make_sound(self) -> str:
        return "Quack!"

class Robot:
    def make_sound(self) -> str:
        return "Beep Boop!"

def animate(entity) -> None:
    """لا تهتم بنوع entity، فقط تحتاج أن تملك make_sound()"""
    print(entity.make_sound())

animate(Duck())    # Quack!
animate(Robot())   # Beep Boop! -- Robot ليست حيواناً إطلاقاً لكنها تعمل!

> 💡 **الفلسفة**: "If it walks like a duck and quacks like a duck, it's a duck" — إذا كان الكائن يملك السلوك المطلوب (method أو attribute)، Python لا تهتم بنوعه الفعلي أو علاقته بأي هرمية وراثة. هذا يختلف جذرياً عن لغات صارمة التنميط مثل Java حيث يجب أن تُعرِّف Interface صريحة. في Python، **السلوك** أهم من **النوع المُعلَن**.

## 14.4 الفئات المجردة (Abstract Base Classes)

### المشكلة: منع إنشاء كائنات من فئة «غير مكتملة»

في مثال `Shape` أعلاه، لا معنى منطقي لإنشاء كائن `Shape()` مباشرة — إنها فكرة عامة، ليست شكلاً حقيقياً محدداً. نريد **إجبار** أي فئة فرعية على تخصيص `area()`، ومنع إنشاء `Shape` مباشرة.

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    from abc import ABC, abstractmethod

    class Shape(ABC):  # ترث من ABC (Abstract Base Class)
        @abstractmethod
        def area(self) -> float:
            """يجب أن تُخصَّص في كل فئة فرعية."""
            pass

        @abstractmethod
        def perimeter(self) -> float:
            """يجب أن تُخصَّص في كل فئة فرعية."""
            pass

        def describe(self) -> str:
            """method عادية - موروثة كما هي، لا حاجة لتخصيصها."""
            return f"{type(self).__name__}: مساحة={self.area():.2f}, محيط={self.perimeter():.2f}"

    class Circle(Shape):
        def __init__(self, radius: float):
            self.radius = radius

        def area(self) -> float:
            return 3.14159 * self.radius ** 2

        def perimeter(self) -> float:
            return 2 * 3.14159 * self.radius

    # shape = Shape()  
    # 💥 TypeError: Can't instantiate abstract class Shape with abstract methods area, perimeter

    circle = Circle(5)
    print(circle.describe())  # Circle: مساحة=78.54, محيط=31.42

    class Square(Shape):
        def __init__(self, side: float):
            self.side = side
        # ⚠️ لو نسينا تعريف area() أو perimeter() هنا:

    # square = Square(5)  
    # 💥 TypeError: Can't instantiate abstract class Square with abstract method area, perimeter
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

> 💡 **نصيحة الخبراء - متى تستخدم Abstract Base Classes؟** عندما تريد تعريف «عقد» (Contract) إلزامي: كل فئة فرعية **يجب** أن تُخصص دوالاً معينة، وإلا لن تستطيع إنشاء كائنات منها أصلاً. هذا يفرض انضباطاً في فرق العمل الكبيرة، ويمنع أخطاءً مثل نسيان تخصيص method ضرورية.

> 🔄 **مقارنة مع لغات أخرى**: هذا المفهوم يُشبه `interface` في Java أو `abstract class` في ++C. الفرق الجوهري أن Python توفرها كمكتبة اختيارية (`abc`) وليس ميزة إلزامية في اللغة، تماشياً مع فلسفة المرونة العامة لـ Python.

## 14.5 Dataclasses: اختصار كتابة الفئات

### المشكلة: التكرار الممل في الفئات البسيطة

In [ ]:
# الطريقة التقليدية - كثير من الكود المتكرر لفئة بسيطة تخزن بيانات فقط
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Point):
            return False
        return self.x == other.x and self.y == other.y

### الحل: `@dataclass`

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p1 = Point(3, 4)
p2 = Point(3, 4)

print(p1)          # Point(x=3, y=4)  -- __repr__ تلقائياً!
print(p1 == p2)     # True  -- __eq__ تلقائياً!

**هذا كل شيء!** `@dataclass` تُولّد تلقائياً `__init__`, `__repr__`, و`__eq__` بناءً على الحقول (Fields) المُعرَّفة بصيغة Type Hints، دون كتابة سطر واحد إضافي.

### قيم افتراضية في Dataclasses

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    age: int = 18  # قيمة افتراضية بسيطة
    grades: list[float] = field(default_factory=list)  # للقيم القابلة للتعديل!

s1 = Student("أحمد")
s1.grades.append(90)
print(s1)  # Student(name='أحمد', age=18, grades=[90])

s2 = Student("سارة")
print(s2)  # Student(name='سارة', age=18, grades=[])  -- مستقلة تماماً!

> ⚠️ **تحذير حرج**: تذكر فخ "mutable default argument" من الفصل العاشر؟ نفس المشكلة تماماً تنطبق هنا! **لا تكتب** `grades: list = []` مباشرة في dataclass:

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    @dataclass
    class Student:
        grades: list = []  # 💥 ValueError: mutable default <class 'list'> is not allowed
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

Python تمنعك صراحة من هذا الخطأ في Dataclasses (رسالة خطأ واضحة!)، وتُجبرك على استخدام `field(default_factory=list)` بدلاً منها، والذي يُنشئ قائمة **جديدة** لكل كائن.

### Methods إضافية في Dataclasses

يمكنك إضافة methods عادية بجانب الحقول:

In [ ]:
from dataclasses import dataclass

@dataclass
class Rectangle:
    width: float
    height: float

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

rect = Rectangle(5, 3)
print(rect)             # Rectangle(width=5, height=3)
print(rect.area())      # 15

### خيارات متقدمة: `frozen` و `order`

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    from dataclasses import dataclass

    @dataclass(frozen=True)  # يجعل الكائن immutable - كالـ tuple!
    class ImmutablePoint:
        x: float
        y: float

    p = ImmutablePoint(3, 4)
    # p.x = 10  
    # 💥 FrozenInstanceError: cannot assign to field 'x'

    @dataclass(order=True)  # يُضيف __lt__, __le__, __gt__, __ge__ تلقائياً
    class Version:
        major: int
        minor: int

    v1 = Version(1, 5)
    v2 = Version(2, 0)
    print(v1 < v2)  # True -- تقارن الحقول بالترتيب المُعرَّف
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

> 💡 **نصيحة الخبراء**: استخدم `frozen=True` عندما تريد كائناً ثابتاً لا يجب تعديله بعد الإنشاء (مثل: إحداثيات، إعدادات ثابتة، مفاتيح لقاموس). هذا يمنع أخطاءً كثيرة ناتجة عن تعديل غير مقصود.

### جدول مقارنة: Class عادية مقابل Dataclass

| الخاصية | Class عادية | `@dataclass` |
|---|---|---|
| `__init__` | يدوي | تلقائي من الحقول |
| `__repr__` | يدوي | تلقائي، منسق بوضوح |
| `__eq__` | يدوي | تلقائي، يقارن كل الحقول |
| التحكم الكامل بالسلوك | كامل | كامل (يمكن إضافة methods) |
| مناسبة لـ | فئات بسلوك معقد ومنطق مخصص | فئات تُركّز بشكل أساسي على تخزين بيانات |

> 🔄 **مقارنة**: `@dataclass` تشبه `record` في Java الحديثة، أو `struct` في بعض اللغات، أو حتى `namedtuple` في Python القديمة (لكن Dataclass أقوى وأكثر مرونة). القاعدة العملية: **إذا كانت فئتك تُركّز أساساً على تجميع بيانات مع القليل من السلوك، استخدم `@dataclass` دائماً** — توفر وقتاً وتقلل الأخطاء بشكل كبير.

## 14.6 مزيد من Magic Methods: `__call__`, `__getitem__`, Context Managers

### `__call__`: جعل الكائن «قابلاً للاستدعاء»

In [ ]:
class Multiplier:
    def __init__(self, factor: float):
        self.factor = factor

    def __call__(self, value: float) -> float:
        return value * self.factor

double = Multiplier(2)
print(double(5))   # 10 -- الكائن نفسه يُستدعى كدالة!
print(double(10))  # 20

### `__getitem__`: دعم الفهرسة `obj[index]`

In [ ]:
class Playlist:
    def __init__(self, songs: list[str]):
        self.songs = songs

    def __getitem__(self, index: int) -> str:
        return self.songs[index]

    def __len__(self) -> int:
        return len(self.songs)

playlist = Playlist(["أغنية 1", "أغنية 2", "أغنية 3"])
print(playlist[0])  # أغنية 1  -- الفهرسة تعمل الآن!

for song in playlist:  # __getitem__ تجعل الكائن قابلاً للتكرار تلقائياً أيضاً!
    print(song)

### `__enter__` و `__exit__`: بناء Context Manager مخصص

تذكر `with open(...) as f:` من الفصل القادم (16)؟ يمكنك بناء كائنات تدعم نفس الصيغة:

In [ ]:
class Timer:
    """context manager يقيس زمن تنفيذ كتلة كود."""

    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        self.end = time.perf_counter()
        print(f"⏱️ استغرق التنفيذ: {self.end - self.start:.4f} ثانية")

with Timer():
    total = sum(i ** 2 for i in range(1_000_000))
    print(f"النتيجة: {total}")

**الناتج:**

```
النتيجة: 333332833333500000
⏱️ استغرق التنفيذ: 0.0842 ثانية
```

> 💡 سنتعمق أكثر في Context Managers في الفصل 16 عند التعامل مع الملفات.

## 14.7 مشروع تطبيقي متكامل: نظام حجز مكتبة

هذا المشروع يجمع كل ما تعلمناه في الفصلين 13 و14: الوراثة، تعدد الأشكال، الفئات المجردة، Dataclasses، وMagic Methods.

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum, auto

# ============ الطبقة 1: نماذج البيانات (Dataclasses) ============

class ItemStatus(Enum):
    """حالة العنصر في المكتبة."""
    AVAILABLE = auto()
    BORROWED = auto()
    RESERVED = auto()

@dataclass
class Member:
    """يمثل عضواً في المكتبة."""
    member_id: int
    name: str
    borrowed_items: list[str] = field(default_factory=list)

    def __str__(self) -> str:
        return f"عضو #{self.member_id}: {self.name}"

# ============ الطبقة 2: الفئة المجردة للعناصر القابلة للاستعارة ============

class LibraryItem(ABC):
    """فئة مجردة أساسية لأي عنصر في المكتبة."""

    def __init__(self, item_id: str, title: str, loan_days: int):
        self.item_id = item_id
        self.title = title
        self.loan_days = loan_days  # عدد الأيام المسموحة للاستعارة
        self.status = ItemStatus.AVAILABLE
        self.borrowed_by: Member | None = None
        self.due_date: datetime | None = None

    @abstractmethod
    def get_category(self) -> str:
        """كل فئة فرعية يجب أن تحدد فئتها."""
        pass

    @abstractmethod
    def display_details(self) -> str:
        """كل فئة فرعية تعرض تفاصيلها الخاصة."""
        pass

    def borrow(self, member: Member) -> bool:
        """استعارة العنصر - سلوك مشترك لكل الأنواع."""
        if self.status != ItemStatus.AVAILABLE:
            print(f"❌ '{self.title}' غير متاح حالياً (الحالة: {self.status.name}).")
            return False

        self.status = ItemStatus.BORROWED
        self.borrowed_by = member
        self.due_date = datetime.now() + timedelta(days=self.loan_days)
        member.borrowed_items.append(self.item_id)

        print(f"✅ استعار {member.name} '{self.title}'. "
              f"الإرجاع المتوقع: {self.due_date.strftime('%Y-%m-%d')}")
        return True

    def return_item(self) -> None:
        """إرجاع العنصر."""
        if self.status != ItemStatus.BORROWED:
            print(f"⚠️ '{self.title}' ليس مُستعاراً أصلاً.")
            return

        if self.borrowed_by:
            self.borrowed_by.borrowed_items.remove(self.item_id)
            print(f"✅ تم إرجاع '{self.title}' من {self.borrowed_by.name}.")

        self.status = ItemStatus.AVAILABLE
        self.borrowed_by = None
        self.due_date = None

    def is_overdue(self) -> bool:
        """التحقق من تأخر الإرجاع - سلوك مشترك."""
        if self.status != ItemStatus.BORROWED or self.due_date is None:
            return False
        return datetime.now() > self.due_date

    def __str__(self) -> str:
        status_display = self.status.name
        return f"[{self.get_category()}] {self.title} ({self.item_id}) - {status_display}"

    def __eq__(self, other) -> bool:
        if not isinstance(other, LibraryItem):
            return False
        return self.item_id == other.item_id

    def __hash__(self) -> int:
        return hash(self.item_id)

# ============ الطبقة 3: الفئات الفرعية (تطبيق تعدد الأشكال) ============

class Book(LibraryItem):
    def __init__(self, item_id: str, title: str, author: str, pages: int):
        super().__init__(item_id, title, loan_days=14)  # الكتب تُستعار 14 يوماً
        self.author = author
        self.pages = pages

    def get_category(self) -> str:
        return "كتاب"

    def display_details(self) -> str:
        return f"📖 {self.title} - تأليف: {self.author} ({self.pages} صفحة)"

class DVD(LibraryItem):
    def __init__(self, item_id: str, title: str, duration_minutes: int):
        super().__init__(item_id, title, loan_days=5)  # الأفلام تُستعار 5 أيام فقط
        self.duration_minutes = duration_minutes

    def get_category(self) -> str:
        return "دي في دي"

    def display_details(self) -> str:
        return f"🎬 {self.title} - المدة: {self.duration_minutes} دقيقة"

class Magazine(LibraryItem):
    def __init__(self, item_id: str, title: str, issue_number: int):
        super().__init__(item_id, title, loan_days=3)  # المجلات 3 أيام فقط
        self.issue_number = issue_number

    def get_category(self) -> str:
        return "مجلة"

    def display_details(self) -> str:
        return f"📰 {self.title} - العدد: {self.issue_number}"

# ============ الطبقة 4: استثناءات مخصصة ============

class LibraryError(Exception):
    """الفئة الأساسية لكل أخطاء نظام المكتبة."""
    pass

class ItemNotFoundError(LibraryError):
    def __init__(self, item_id: str):
        super().__init__(f"العنصر '{item_id}' غير موجود في المكتبة.")

class MemberNotFoundError(LibraryError):
    def __init__(self, member_id: int):
        super().__init__(f"العضو رقم {member_id} غير موجود.")

# ============ الطبقة 5: النظام الرئيسي ============

class Library:
    """يُدير كل عمليات المكتبة: الأعضاء، العناصر، الاستعارة، الإرجاع."""

    def __init__(self, name: str):
        self.name = name
        self.items: dict[str, LibraryItem] = {}
        self.members: dict[int, Member] = {}

    # --- إدارة العناصر ---
    def add_item(self, item: LibraryItem) -> None:
        self.items[item.item_id] = item
        print(f"📚 أُضيف '{item.title}' إلى مكتبة '{self.name}'.")

    def find_item(self, item_id: str) -> LibraryItem:
        try:
            return self.items[item_id]
        except KeyError:
            raise ItemNotFoundError(item_id)

    # --- إدارة الأعضاء ---
    def register_member(self, member: Member) -> None:
        self.members[member.member_id] = member
        print(f"👤 تم تسجيل {member.name} كعضو جديد.")

    def find_member(self, member_id: int) -> Member:
        try:
            return self.members[member_id]
        except KeyError:
            raise MemberNotFoundError(member_id)

    # --- العمليات الأساسية ---
    def borrow_item(self, item_id: str, member_id: int) -> None:
        item = self.find_item(item_id)
        member = self.find_member(member_id)
        item.borrow(member)

    def return_item(self, item_id: str) -> None:
        item = self.find_item(item_id)
        item.return_item()

    # --- تقارير (تطبيق تعدد الأشكال بوضوح) ---
    def list_all_items(self) -> None:
        print(f"\n📋 كل عناصر مكتبة '{self.name}':")
        print("-" * 60)
        for item in self.items.values():
            print(item.display_details())  # كل نوع يعرض تفاصيله الخاصة!
        print("-" * 60)

    def list_available_items(self) -> list[LibraryItem]:
        return [item for item in self.items.values()
                if item.status == ItemStatus.AVAILABLE]

    def list_overdue_items(self) -> list[LibraryItem]:
        return [item for item in self.items.values() if item.is_overdue()]

    def items_by_category(self) -> dict[str, int]:
        """عدد العناصر حسب الفئة - باستخدام polymorphism (get_category)."""
        counts: dict[str, int] = {}
        for item in self.items.values():
            category = item.get_category()
            counts[category] = counts.get(category, 0) + 1
        return counts

# ============ التشغيل التجريبي ============

if __name__ == "__main__":
    library = Library("مكتبة الأمل العامة")

    # إضافة عناصر متنوعة (تعدد الأشكال في العمل)
    library.add_item(Book("B001", "بايثون للمحترفين", "محمد أحمد", 350))
    library.add_item(Book("B002", "Clean Code", "Robert Martin", 400))
    library.add_item(DVD("D001", "رحلة إلى الفضاء", 120))
    library.add_item(Magazine("M001", "التقنية اليوم", 45))

    # تسجيل أعضاء
    library.register_member(Member(1, "أحمد"))
    library.register_member(Member(2, "سارة"))

    print()
    library.list_all_items()

    print("\n--- عمليات الاستعارة ---")
    library.borrow_item("B001", 1)
    library.borrow_item("D001", 2)

    try:
        library.borrow_item("B001", 2)  # محاولة استعارة كتاب مُستعار بالفعل
    except LibraryError as e:
        print(f"❌ {e}")

    try:
        library.borrow_item("X999", 1)  # عنصر غير موجود
    except ItemNotFoundError as e:
        print(f"❌ {e}")

    print("\n--- حالة المكتبة ---")
    library.list_all_items()

    available = library.list_available_items()
    print(f"\n✅ العناصر المتاحة حالياً: {len(available)}")
    for item in available:
        print(f"  - {item}")

    print(f"\n📊 توزيع العناصر حسب الفئة: {library.items_by_category()}")

    print("\n--- إرجاع عنصر ---")
    library.return_item("B001")

    library.list_all_items()

**الناتج المتوقع (مختصر):**

```
📚 أُضيف 'بايثون للمحترفين' إلى مكتبة 'مكتبة الأمل العامة'.
📚 أُضيف 'Clean Code' إلى مكتبة 'مكتبة الأمل العامة'.
📚 أُضيف 'رحلة إلى الفضاء' إلى مكتبة 'مكتبة الأمل العامة'.
📚 أُضيف 'التقنية اليوم' إلى مكتبة 'مكتبة الأمل العامة'.
👤 تم تسجيل أحمد كعضو جديد.
👤 تم تسجيل سارة كعضو جديد.

📋 كل عناصر مكتبة 'مكتبة الأمل العامة':
------------------------------------------------------------
📖 بايثون للمحترفين - تأليف: محمد أحمد (350 صفحة)
📖 Clean Code - تأليف: Robert Martin (400 صفحة)
🎬 رحلة إلى الفضاء - المدة: 120 دقيقة
📰 التقنية اليوم - العدد: 45
------------------------------------------------------------

--- عمليات الاستعارة ---
✅ استعار أحمد 'بايثون للمحترفين'. الإرجاع المتوقع: 2025-01-29
✅ استعار سارة 'رحلة إلى الفضاء'. الإرجاع المتوقع: 2025-01-20
❌ 'بايثون للمحترفين' غير متاح حالياً (الحالة: BORROWED).
❌ العنصر 'X999' غير موجود في المكتبة.

--- حالة المكتبة ---
📋 كل عناصر مكتبة 'مكتبة الأمل العامة':
------------------------------------------------------------
📖 بايثون للمحترفين - تأليف: محمد أحمد (350 صفحة)
📖 Clean Code - تأليف: Robert Martin (400 صفحة)
🎬 رحلة إلى الفضاء - المدة: 120 دقيقة
📰 التقنية اليوم - العدد: 45
------------------------------------------------------------

✅ العناصر المتاحة حالياً: 2
  - [كتاب] Clean Code (B002) - AVAILABLE
  - [مجلة] التقنية اليوم (M001) - AVAILABLE

📊 توزيع العناصر حسب الفئة: {'كتاب': 2, 'دي في دي': 1, 'مجلة': 1}

--- إرجاع عنصر ---
✅ تم إرجاع 'بايثون للمحترفين' من أحمد.

📋 كل عناصر مكتبة 'مكتبة الأمل العامة':
------------------------------------------------------------
📖 بايثون للمحترفين - تأليف: محمد أحمد (350 صفحة)
📖 Clean Code - تأليف: Robert Martin (400 صفحة)
🎬 رحلة إلى الفضاء - المدة: 120 دقيقة
📰 التقنية اليوم - العدد: 45
------------------------------------------------------------
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**تحليل التصميم**: لاحظ كيف أن:
- `LibraryItem` فئة مجردة تفرض على كل نوع فرعي تعريف `get_category()` و`display_details()`
- السلوك المشترك (`borrow`, `return_item`, `is_overdue`) مكتوب **مرة واحدة فقط** في الفئة الأساسية
- `Library.list_all_items()` لا تحتاج معرفة نوع كل عنصر — **تعدد الأشكال** يتكفل بذلك تلقائياً
- الاستثناءات المخصصة (`ItemNotFoundError`, `MemberNotFoundError`) تجعل معالجة الأخطاء دقيقة وواضحة

<details>
<summary>✅ تحدي إضافي: إضافة نظام غرامات التأخير (الحل)</summary>

```python
class LibraryItem(ABC):
    # ... (الكود السابق كما هو) ...
    FINE_PER_DAY = 2.0  # ريال لكل يوم تأخير

    def calculate_fine(self) -> float:
        """حساب الغرامة إذا كان العنصر متأخراً."""
        if not self.is_overdue() or self.due_date is None:
            return 0.0
        days_late = (datetime.now() - self.due_date).days
        return days_late * self.FINE_PER_DAY

# استخدام:
item = library.find_item("B001")
fine = item.calculate_fine()
if fine > 0:
    print(f"⚠️ غرامة تأخير: {fine} ر.س")
```

🧾 **للقراءة فقط** — مقتطف يضيف إلى صنف/دالة سبق تعريفها، وليس وحدة مستقلة.

</details>

## 14.8 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1**: أنشئ فئة أساسية `Shape` بـ method `area()` تُرجع 0 افتراضياً، وفئة فرعية `Square(Shape)` تُخصص `area()` لحساب مساحة المربع.

**تمرين 2**: حوّل فئة `Point` (بحقلين x, y) إلى `@dataclass`، واطبع كائنين للمقارنة.

<details>
<summary>✅ الحل</summary>

In [ ]:
# تمرين 1
class Shape:
    def area(self) -> float:
        return 0

class Square(Shape):
    def __init__(self, side: float):
        self.side = side

    def area(self) -> float:
        return self.side ** 2

sq = Square(4)
print(sq.area())  # 16

# تمرين 2
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1)          # Point(x=1, y=2)
print(p1 == p2)     # True

</details>

### 🟡 مستوى متوسط

**تمرين 3**: أنشئ فئة مجردة `PaymentMethod` بـ abstractmethod باسم `process(amount)`. أنشئ فئتين فرعيتين: `CreditCard` و`PayPal`، كل منهما تطبع رسالة مختلفة عند `process()`.

**تمرين 4**: أنشئ `@dataclass` باسم `Product` بحقول `name`, `price`, `quantity=1`. أضف method `total()` تحسب `price * quantity`. استخدم `frozen=True`.

<details>
<summary>✅ الحل</summary>

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    # تمرين 3
    from abc import ABC, abstractmethod

    class PaymentMethod(ABC):
        @abstractmethod
        def process(self, amount: float) -> None:
            pass

    class CreditCard(PaymentMethod):
        def process(self, amount: float) -> None:
            print(f"💳 تمت معالجة {amount} ر.س عبر البطاقة الائتمانية.")

    class PayPal(PaymentMethod):
        def process(self, amount: float) -> None:
            print(f"🅿️ تمت معالجة {amount} ر.س عبر PayPal.")

    methods: list[PaymentMethod] = [CreditCard(), PayPal()]
    for method in methods:
        method.process(100)

    # تمرين 4
    from dataclasses import dataclass

    @dataclass(frozen=True)
    class Product:
        name: str
        price: float
        quantity: int = 1

        def total(self) -> float:
            return self.price * self.quantity

    p = Product("لابتوب", 3500, 2)
    print(p.total())  # 7000
    # p.price = 4000  # 💥 FrozenInstanceError
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

</details>

### 🔴 تحدي

**تمرين 5**: صمم نظام موظفين: فئة مجردة `Employee` بـ abstractmethod `calculate_salary()`. فئتان فرعيتان: `FullTimeEmployee` (راتب ثابت شهري) و`FreelanceEmployee` (أجر بالساعة × عدد الساعات). أضف فئة `Payroll` تحسب إجمالي رواتب قائمة موظفين مختلطة النوع (تطبيق تعدد الأشكال).

<details>
<summary>✅ الحل</summary>

In [ ]:
from abc import ABC, abstractmethod

class Employee(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def calculate_salary(self) -> float:
        pass

    def __str__(self) -> str:
        return f"{self.name}: {self.calculate_salary():.2f} ر.س"

class FullTimeEmployee(Employee):
    def __init__(self, name: str, monthly_salary: float):
        super().__init__(name)
        self.monthly_salary = monthly_salary

    def calculate_salary(self) -> float:
        return self.monthly_salary

class FreelanceEmployee(Employee):
    def __init__(self, name: str, hourly_rate: float, hours_worked: float):
        super().__init__(name)
        self.hourly_rate = hourly_rate
        self.hours_worked = hours_worked

    def calculate_salary(self) -> float:
        return self.hourly_rate * self.hours_worked

class Payroll:
    def __init__(self):
        self.employees: list[Employee] = []

    def add_employee(self, employee: Employee) -> None:
        self.employees.append(employee)

    def total_payroll(self) -> float:
        return sum(emp.calculate_salary() for emp in self.employees)

    def print_report(self) -> None:
        print("📋 تقرير الرواتب:")
        for emp in self.employees:
            print(f"  {emp}")
        print(f"الإجمالي: {self.total_payroll():.2f} ر.س")

payroll = Payroll()
payroll.add_employee(FullTimeEmployee("أحمد", 8000))
payroll.add_employee(FreelanceEmployee("سارة", 50, 120))
payroll.print_report()

# 📋 تقرير الرواتب:
#   أحمد: 8000.00 ر.س
#   سارة: 6000.00 ر.س
# الإجمالي: 14000.00 ر.س

</details>

## 14.9 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما وظيفة super() وأين يجب استدعاؤها عادة؟</summary>

**الإجابة**: `super()` تُتيح الوصول لـ methods الفئة الأساسية (Parent) من داخل الفئة الفرعية (Child). عادة تُستدعى `super().__init__(...)` كأول سطر في `__init__` الفئة الفرعية لضمان تهيئة كل الخصائص الموروثة من الفئة الأساسية قبل إضافة أي خصائص جديدة خاصة بالفئة الفرعية.
</details>

<details>
<summary>❓ السؤال 2: ما الفرق بين isinstance() و type() == عند التحقق من نوع الكائن؟</summary>

**الإجابة**: `isinstance(obj, Class)` تأخذ الوراثة بعين الاعتبار — تُرجع `True` إذا كان الكائن من تلك الفئة **أو من أي فئة فرعية منها**. بينما `type(obj) == Class` تتحقق من التطابق **الدقيق** فقط، وتُرجع `False` حتى لو كان الكائن من فئة فرعية. `isinstance()` هي الطريقة الموصى بها دائماً في Python.
</details>

<details>
<summary>❓ السؤال 3: لماذا لا يمكن إنشاء كائن مباشرة من فئة مجردة (Abstract Class)؟</summary>

**الإجابة**: لأن الفئة المجردة تحتوي على `@abstractmethod` واحدة أو أكثر بدون تنفيذ فعلي — إنها تُعرّف فقط «ما يجب أن تفعله» الفئات الفرعية، وليس «كيف». محاولة إنشاء كائن منها مباشرة تُطلق `TypeError`، لأن Python تمنع إنشاء كائنات من فئات «غير مكتملة» السلوك.
</details>

<details>
<summary>❓ السؤال 4: ما الفائدة الرئيسية من @dataclass مقارنة بكتابة الفئة يدوياً؟</summary>

**الإجابة**: `@dataclass` تُولّد تلقائياً `__init__`, `__repr__`, و`__eq__` بناءً على الحقول المُعرَّفة، دون الحاجة لكتابة أي من هذه الدوال يدوياً. هذا يختصر الكود بشكل كبير في الفئات التي تُركّز أساساً على تخزين بيانات، ويقلل احتمال الأخطاء الناتجة عن الكتابة اليدوية المتكررة.
</details>

<details>
<summary>❓ السؤال 5: اشرح مفهوم تعدد الأشكال (Polymorphism) بمثال من المشروع التطبيقي.</summary>

**الإجابة**: تعدد الأشكال يعني أن كائنات من فئات مختلفة (مثل `Book`, `DVD`, `Magazine`) يمكن التعامل معها بواجهة موحدة (مثل استدعاء `item.display_details()` على أي منها)، لكن كل نوع يُنفّذ السلوك بطريقته الخاصة. في مشروع المكتبة، `Library.list_all_items()` تستدعي `display_details()` على كل عنصر دون معرفة نوعه الفعلي، والنتيجة الصحيحة تظهر تلقائياً لكل نوع.
</details>

## 14.10 ملخص الفصل (Cheat Sheet)

```python
# ============ الوراثة ============
class Parent:
    def __init__(self, x):
        self.x = x
    def greet(self):
        return f"Parent: {self.x}"

class Child(Parent):
    def __init__(self, x, y):
        super().__init__(x)    # استدعاء __init__ الأساسي
        self.y = y
    def greet(self):
        return f"{super().greet()}, Child: {self.y}"  # تمديد السلوك

# ============ تعدد الأشكال ============
for obj in [ClassA(), ClassB(), ClassC()]:
    obj.common_method()  # كل فئة تُنفّذ بطريقتها

# ============ الفئات المجردة ============
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def required_method(self): ...

# Base()  # 💥 TypeError - لا يمكن إنشاء كائن مباشرة

# ============ Dataclasses ============
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    tags: list = field(default_factory=list)  # ⚠️ ليس []

@dataclass(frozen=True)   # immutable
@dataclass(order=True)    # يدعم < <= > >=

# ============ Magic Methods إضافية ============
def __call__(self, *args): ...      # obj(args)
def __getitem__(self, i): ...       # obj[i]
def __enter__(self): ...            # with obj as x:
def __exit__(self, *args): ...

# ============ isinstance ============
isinstance(obj, SomeClass)          # ✅ يأخذ الوراثة بعين الاعتبار
type(obj) == SomeClass              # ❌ تطابق دقيق فقط
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

| المفهوم | الاستخدام الأمثل |
|---|---|
| وراثة بسيطة | عندما يوجد «هو نوع من» (Is-A) واضح: Dog is-a Animal |
| Composition | عندما يوجد «يملك» (Has-A): Car has-a Engine (بديل أنظف أحياناً من وراثة معقدة) |
| Abstract Base Class | فرض «عقد» إلزامي على كل فئة فرعية |
| Dataclass | فئات تُركّز على تخزين بيانات مع منطق بسيط |
| Duck Typing | عندما يهمك السلوك المتاح أكثر من النوع الرسمي |

## 14.11 ماذا بعد؟

لقد أكملت الآن **الجزء الثاني بالكامل**: البرمجة الوظيفية والكائنية. تمتلك فهماً عميقاً لكيفية تنظيم الكود بطرق متقدمة، من Decorators وGenerators إلى الوراثة وتعدد الأشكال والـ Dataclasses.

لكن حتى الآن، اعتمدنا بشكل كبير على التجربة والخطأ لمعرفة أنواع البيانات المتوقعة في دوالنا. ماذا لو استطعت أن **تُصرّح** بوضوح لـ Python (ولزملائك، ولأدوات التحقق التلقائي) بالضبط ما هي الأنواع المتوقعة في كل مكان، وتكتشف الأخطاء **قبل** تشغيل الكود أصلاً؟

في **الفصل الخامس عشر: Type Hints - البرمجة الواضحة**، سنتعمق أكثر فيما بدأناه منذ الفصل الثالث: Type Hints المتقدمة، أداة `mypy` للتحقق التلقائي، وكيف تجعل Type Hints كودك أكثر أماناً ووضوحاً واحترافية — ممارسة أصبحت معياراً في كل مشاريع Python الحديثة والكبرى.

---

### الخطوة التالية

- [الفصل الخامس عشر: Type Hints — نحو برمجة أكثر وضوحًا ودقة (باستخدام mypy)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-15.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الرابع عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*